# yt-clips Colab Worker
## Runtime -> T4 GPU
Run all cells. Worker listens for jobs from your Mac.
Add host reference photos to the `photos/` folder for automatic face detection & tracking.

In [ ]:
# CELL 1: Mount Drive + Load API Key
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import os
from pathlib import Path

for p in ["/content/drive/MyDrive/yt-clips", "/content/drive/My Drive/yt-clips"]:
    if Path(p).exists():
        os.chdir(p)
        print(f"OK: {p}")
        break
else:
    os.chdir("/content")

try:
    from google.colab import userdata
    key = userdata.get("GOOGLE_API_KEY") or userdata.get("AI_API_KEY")
    if key:
        os.environ["GOOGLE_API_KEY"] = key
        os.environ["AI_API_KEY"] = key
        print("API key loaded from secrets")
except:
    print("No API key in secrets. Gemini rate limits low.")

In [ ]:
# CELL 2: Install Dependencies
import subprocess, sys

def run(cmd):
    subprocess.run(cmd, shell=True, capture_output=True)

print("Installing system deps...")
run("apt-get install -y -qq aria2 ffmpeg > /dev/null 2>&1")
run("npm install -g localtunnel > /dev/null 2>&1")
run("curl -fsSL https://deno.land/x/install/install.sh | sh > /dev/null 2>&1")
os.environ["PATH"] += ":/root/.deno/bin"

print("Installing Python deps...")
run("pip install -q yt-dlp faster-whisper rich PyYAML opencv-python-headless numpy "
     "filterpy scipy google-genai google-generativeai openai python-dotenv "
     "ultralytics torch face_recognition --extra-index-url https://download.pytorch.org/whl/cu121")

gpu = subprocess.run("nvidia-smi --query-gpu=name --format=csv,noheader 2>/dev/null",
                     shell=True, capture_output=True, text=True).stdout.strip()
print(f"GPU: {gpu or 'NONE! Use T4'}")

In [ ]:
# CELL 3: Start Worker + Tunnel
import time

for folder in ["input", "temp", "transcripts", "highlights", "shorts", "logs"]:
    Path(folder).mkdir(exist_ok=True)

subprocess.run("pkill -f 'python watcher.py' 2>/dev/null || true", shell=True)
subprocess.run("pkill -f 'lt --port' 2>/dev/null || true", shell=True)
time.sleep(1)

subprocess.Popen([sys.executable, "watcher.py"], stdout=open("watcher.log","w"), stderr=subprocess.STDOUT)
time.sleep(2)
subprocess.Popen(["lt", "--port", "5000"], stdout=open("tunnel.log","w"), stderr=subprocess.STDOUT)
time.sleep(5)

with open("tunnel.log") as f:
    for line in f:
        if "://" in line.strip():
            with open("colab_url.txt", "w") as out:
                out.write(line.strip())
            print(f"Tunnel URL: {line.strip()}")
            break

In [ ]:
# CELL 4: Monitor (keep this cell running)
print("=" * 40)
print("  WORKER IS ONLINE!")
print("=" * 40)
print("On Mac, run:")
print('  ./automate.sh "URL"')
print("  -> Option 2 (Remote Run)")

try:
    last_pos = 0
    while True:
        time.sleep(10)
        if Path("watcher.log").exists():
            with open("watcher.log", "r") as f:
                f.seek(0, 2)
                if f.tell() < last_pos:
                    last_pos = 0
                f.seek(last_pos)
                for l in f.readlines():
                    if l.strip():
                        print(l.strip())
                last_pos = f.tell()
except KeyboardInterrupt:
    print("Stopped.")